# TS_4. Metrics, baselines, validation

In [1]:
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.forecasting.theta import ThetaModel
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings("ignore")

In [2]:
rng = np.random.default_rng(0)
t = np.arange(730)
y = 100 + 0.05 * t + np.array([0, 4, 6, 5, 9, 18, -12])[t % 7] + rng.normal(0, 4, 730)
S, H = 7, 7

In [3]:
def naive(tr, h): return np.repeat(tr[-1], h)
def snaive(tr, h): return np.tile(tr[-S:], h // S + 1)[:h]
def mean(tr, h): return np.repeat(tr.mean(), h)
def drift(tr, h): return tr[-1] + (tr[-1] - tr[0]) / (len(tr) - 1) * np.arange(1, h + 1)
def theta(tr, h): return np.asarray(ThetaModel(tr, period=S, deseasonalize=True, use_test=False).fit().forecast(h))

models = {"Naive": naive, "SNaive": snaive, "Mean": mean, "Drift": drift, "Theta": theta}

In [4]:
def metrics(yt, yp, tr):
    e = yt - yp
    return {"MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
            "MAPE": 100 * np.abs(e / yt).mean(), "sMAPE": 100 * (2 * np.abs(e) / (np.abs(yt) + np.abs(yp))).mean(),
            "MASE": np.abs(e).mean() / np.abs(tr[S:] - tr[:-S]).mean()}

def backtest(y, f, scheme="expanding", n_folds=10, window=112):
    res = []
    for k in range(n_folds):
        start = len(y) - (n_folds - k) * H
        tr = y[:start] if scheme == "expanding" else y[start - window:start]
        res.append(metrics(y[start:start + H], f(tr, H), tr))
    return pd.DataFrame(res).mean()

In [5]:
table = pd.DataFrame({(s, n): backtest(y, f, s) for s in ("expanding", "sliding") for n, f in models.items()}).T
table.round(3)

MAE    RMSE    MAPE   sMAPE   MASE
expanding Naive    8.031  10.199   5.852   5.837  1.795
          SNaive   4.149   5.145   2.961   2.970  0.928
          Mean    18.118  20.032  12.648  13.680  4.050
          Drift    8.027  10.220   5.856   5.832  1.794
          Theta    3.463   4.197   2.488   2.490  0.774
sliding   Naive    8.031  10.199   5.852   5.837  1.775
          SNaive   4.149   5.145   2.961   2.970  0.918
          Mean     8.067   9.851   5.805   5.858  1.784
          Drift    8.082  10.265   5.894   5.874  1.786
          Theta    3.490   4.167   2.512   2.514  0.773

In [6]:
# intervals for Seasonal Naive from backtest residuals
errs = np.array([y[s:s + H] - snaive(y[:s], H) for s in range(500, 723)])
lo, hi = np.quantile(errs[:150], [0.025, 0.975], axis=0)
inside = (errs[150:] >= lo) & (errs[150:] <= hi)
pd.DataFrame({"coverage": inside.mean(axis=0), "width": hi - lo}, index=range(1, H + 1))

,coverage,width
1,0.945205,19.961750
2,0.945205,19.961750
3,0.945205,19.961750
4,0.958904,20.746311
5,0.958904,20.746311
6,0.958904,20.746311
7,0.958904,20.746311


In [7]:
# leakage: random split vs time split
d = pd.DataFrame({"y": y})
for l in (1, 7, 14):
    d[f"lag{l}"] = d["y"].shift(l)
d["t"] = t
d = d.dropna()
X, yy = d.drop(columns="y").values, d["y"].values

def cv(splits):
    rf = RandomForestRegressor(100, random_state=0)
    return np.mean([np.abs(yy[te] - rf.fit(X[tr], yy[tr]).predict(X[te])).mean() for tr, te in splits])

print("random split MAE:", round(cv(KFold(5, shuffle=True, random_state=0).split(X)), 3))
print("TimeSeriesSplit MAE:", round(cv(TimeSeriesSplit(5, gap=7).split(X)), 3))

random split MAE: 4.19


TimeSeriesSplit MAE: 4.519


In [8]:
# purged K-fold with embargo
def purged_kfold(n, k=5, horizon=7, embargo=7):
    idx = np.arange(n)
    for te in np.array_split(idx, k):
        keep = (idx + horizon - 1 < te[0]) | (idx > te[-1] + embargo)
        yield idx[keep], te

print("Purged K-Fold MAE:", round(cv(purged_kfold(len(X))), 3))

Purged K-Fold MAE: 4.342
